In [1]:
import dask
from dask_jobqueue import SLURMCluster
import dask.dataframe as dd
from pathlib import Path
import os
import pystac_client
import planetary_computer
from datetime import timedelta
import json
from shapely.geometry import box,shape, MultiPoint
import geopandas as gpd
import pandas as pd

/home/ksb781/miniconda3/envs/mpc/lib/python3.11/site-packages/dask/dataframe/_pyarrow_compat.py:17: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 11.0.0. Please consider upgrading.
  warnings.warn(


In [2]:
cluster = SLURMCluster(cores=32, memory='64 GB', processes=1, scheduler_options={"dashboard_address": ":45578"}, log_directory='logs')
cluster.scale(1)

In [3]:
from dask.distributed import Client
client = Client(cluster)

In [16]:
cluster.close()

## Add date, track_number columns & save as parquet

In [ ]:
from pathlib import Path
import dask
import dask.dataframe as dd
import geopandas as gpd
import re
from datetime import datetime, timedelta
import json
from shapely.geometry import shape
import dask_geopandas as dgd
import ipdb

def getDate(year, doy):
    start_of_year = datetime(int(year), 1, 1)
    calendar_date = start_of_year + timedelta(days=int(doy) - 1)
    return calendar_date.strftime('%Y-%m-%d')


def addTrackNumberForFile(csvFile):
    search = re.search(r'(\d{4})(\d{3})', csvFile.stem)
    df = dd.read_csv(csvFile, dtype={'system:index': 'object'})
    df['date'] = getDate(search.group(1), search.group(2))
    df['track_id'] = re.search(r'(\d{13})_O(\d{5})_.*_T(\d{5})',
                               csvFile.stem)[0]

    df['geometry'] = df['.geo'].apply(lambda x: shape(json.loads(x)),
                                      meta=('geometry', object))
    df = df.drop('.geo', axis=1)
    ddf = dgd.from_dask_dataframe(df).compute()
    ddf.to_parquet(csvFile.with_suffix('.parquet'))
    return


def addTrackNumber():
    dataFolder = Path.home() / 'GEDI2019'
    csvFiles = dataFolder.glob('**/GEDI02*.csv')
    res = []
    for file in csvFiles:
        res.append(dask.delayed(addTrackNumberForFile)(file))
    dask.compute(res)

task = client.submit(addTrackNumber)

In [ ]:
task

In [ ]:
client.restart()

In [ ]:
task.result()

## check if GEE manage each track as an asset

In [ ]:
def checkFilenames(folder):
    track_numbers = []
    for file in folder.glob('*.parquet'):
        num = re.search(r'_T(\d+)' ,file.stem).group(1)
        if num in track_numbers:
            print(f'Track number {num} is not unique, found in {folder.stem}')
            break
        track_numbers.append(num)

def checkParquetIdUnique():
    dataFolder = Path.home() / 'GEDI2019'
    df = dd.read_parquet(dataFolder)
    npid = df.parquet_id.nunique().compute()
    nfiles = len(set(list(dataFolder.glob('**/*.parquet'))))
    print(f'Number of unique parquet ids: {npid}')
    print(f'Number of unique parquet files: {nfiles}')
    
    

The parquet_id can indentify a unique parquet file.

In [ ]:

def checkParquetIdUnique():
    dataFolder = Path.home() / 'GEDI2019'
    df = dd.read_parquet(dataFolder)
    npid = df.parquet_id.nunique().compute()
    nfiles = len(set([f.stem for f in dataFolder.glob('**/*.parquet')]))
    print(f'Number of unique parquet ids: {npid}')
    print(f'Number of unique parquet files: {nfiles}')
    
    

# S2 download

For each MGRS zone,
    - Dealing with each parquet(csv) file, because points from each file are obtained from the same time. 
    But they might have different growing season? -> use the union to filter 

api.search item columns
Index(['geometry', 'datetime', 'platform', 'proj:epsg', 'instruments',
       's2:mgrs_tile', 'constellation', 's2:granule_id', 'eo:cloud_cover',
       's2:datatake_id', 's2:product_uri', 's2:datastrip_id',
       's2:product_type', 'sat:orbit_state', 's2:datatake_type',
       's2:generation_time', 'sat:relative_orbit', 's2:water_percentage',
       's2:mean_solar_zenith', 's2:mean_solar_azimuth',
       's2:processing_baseline', 's2:snow_ice_percentage',
       's2:vegetation_percentage', 's2:thin_cirrus_percentage',
       's2:cloud_shadow_percentage', 's2:nodata_pixel_percentage',
       's2:unclassified_percentage', 's2:dark_features_percentage',
       's2:not_vegetated_percentage', 's2:degraded_msi_data_percentage',
       's2:high_proba_clouds_percentage', 's2:reflectance_conversion_factor',
       's2:medium_proba_clouds_percentage',
       's2:saturated_defective_pixel_percentage'],
      dtype='object')
asset id: S2B_MSIL2A_20190424T213759_R100_T01GEM_20201105T174005

In [8]:
from s2_download import S2Downloader
s2downloader = S2Downloader("GEDI2019", debug=False)
# task = client.submit(s2downloader.get_s2_for_zone, '01G')
ll = s2downloader.get_s2_for_zone('01G')
# task = client.map(s2downloader.download_zone, ['32M', '56H'])


RuntimeError: P2P shuffling c901054ff3803e5927dbd9392a87f937 failed during transfer phase

In [7]:
client.restart()

<Client: 'tcp://10.84.3.176:45677' processes=0 threads=0, memory=0 B>

In [14]:
task

<Future: error, key: get_s2_for_zone-ce09c41e264f8ae243666f502e2b6ec7>

In [15]:
task.result()

RuntimeError: P2P shuffling 588cd4806a69cf704dd7a9515caea5c6 failed during transfer phase

In [ ]:
cluster.close()

In [ ]:
client

## read parquet & search dask dataframe


In [ ]:
def getItem():
    item_id = "S2B_MSIL2A_20190424T213759_R100_T01GEM_20201105T174005"
    api = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                modifier=planetary_computer.sign_inplace)
    asset = api.get_collection("sentinel-2-l2a").assets["geoparquet-items"]
    s2l2a = dd.read_parquet(
            asset.href, storage_options=asset.extra_fields["table:storage_options"]
        )
    search = s2l2a.query('id=="S2B_MSIL2A_20190424T213759_R100_T01GEM_20201105T174005"')
    # search = s2l2a.head()
    item = search.compute()
    print(item)
    
task = client.submit(getItem)

In [ ]:
task

In [ ]:
task.result()

In [ ]:
client.recreate_error_locally(task)

In [ ]:
def plotHistogram():
    import matplotlib.pyplot as plt
    import dask.array as da
    import numpy as np
    dataFolder = Path('/home/ksb781/GEDI2019')
    ddf = dd.read_csv(dataFolder / f'*/*.csv', usecols=['rh98', 'pft_class', '.geo'], blocksize=64e6)
    h, bins = da.histogram(ddf['rh98'], bins=np.arange(60))
    print(h, bins)
    plt.stairs(h, bins)
    plt.savefig('output/rh98.png')

    print('plot saved')

In [ ]:
def countZero():
    import matplotlib.pyplot as plt
    import dask.dataframe as dd
    import dask.array as da
    import numpy as np
    dataFolder = Path('/home/ksb781/GEDI2019')
    keyRhs = ['rh10', 'rh25', 'rh50', 'rh75', 'rh90', 'rh98']
    ddf = dd.read_csv(dataFolder / f'*/*.csv', usecols=keyRhs, blocksize=64e6)
    for key in keyRhs:
        h, bins = da.histogram(ddf[key], bins=np.arange(60))
        print(h, bins)
        plt.stairs(h, bins)
        plt.savefig(f'output/{key}.png')
        count = ddf[key].value_counts()
        count.to_csv(f'output/value_counts_{key}.csv')

    print('done')

In [ ]:
task = client.submit(countZero)

In [ ]:
client.cancel(task)

In [ ]:
task

In [ ]:
client.close()

In [ ]:
cluster.close()